**Navigation** : [Index](README.md)

# Notebook de conception de Notebook

Ce Notebook .Net interactive a pour objectif de permettre la création assistée d'autres notebooks .Net interactive en confiant le soin à ChatGPT d'analyser et de proposer des modifications d'une version courante, et en prenant en charge la mise à jour et l'exécution des mises à jour en function calling Open AI grâce à l'API .Net interactive. 


### 1. Initialisation

On installe des packages pour la manipulation de notebook et pour l'orchestration de LLMs.

In [1]:
// #r "nuget: Microsoft.DotNet.Interactive, *-*"
#r "nuget: Microsoft.DotNet.Interactive.CSharp, *-*"
#r "nuget: Microsoft.DotNet.Interactive.Documents, *-*"
#r "nuget: Microsoft.DotNet.Interactive.PackageManagement, *-*"
#r "nuget: Microsoft.Extensions.Logging"
#r "nuget: Microsoft.SemanticKernel, *-*"
#r "nuget: Microsoft.SemanticKernel.Planners.OpenAI, *-*"
#r "nuget: Microsoft.SemanticKernel.Agents.Core, *-*"

### Importation des fichiers `.cs`

Un certain nombre de classes ont été conçues pour fournir l'exécution et la mise à jour de notebook .Net interactive à l'aide de Semantic-Kernel.

In [2]:
// Load some helper functions to load values from settings.json
#!import ../../Config/Settings.cs 

#!import DisplayLogger.cs
#!import DisplayLoggerProvider.cs
#!import NotebookExecutor.cs
// #!import WorkbookInteractionBase.cs
// #!import WorkbookUpdateInteraction.cs
// #!import WorkbookValidation.cs
// #!import NotebookUpdaterBase.cs
// #!import AutoInvokeSKAgentsNotebookUpdater.cs

// #!import NotebookPlannerUpdater.cs
// #!import AutoGenNotebookUpdater.cs


- **Imports des espaces de noms**

On prend soin de distinguer le kernel d'exécution de notebook .Net interactive, et le kernel de semantic-kernel.

In [3]:
using Microsoft.DotNet.Interactive;
using Microsoft.SemanticKernel;
using Microsoft.SemanticKernel.Planning;
using Microsoft.SemanticKernel.Connectors.OpenAI;
  
using System;
using System.IO;
using System.Threading.Tasks;

using Microsoft.Extensions.Logging;
using Microsoft.Extensions.DependencyInjection;

using SKernel = Microsoft.SemanticKernel.Kernel;
using IKernel = Microsoft.DotNet.Interactive.Kernel;

### 1. Mode de Fourniture des Informations

On permet à l'utilisateur de saisir les informations décrivant la tâche à accomplir dans le notebook de travail de plusieurs façons différentes.

In [4]:
public enum InformationMode
{
    Variable,
    Prompt,
    File
}



### Exercice : Ajouter un mode de fourniture URL

L'enum `InformationMode` definit actuellement trois modes (Variable, Prompt, File). Vous pouvez ajouter un quatrieme mode pour recuperer la description depuis une URL distante.

**Objectif** : Completez le code ci-dessous pour ajouter le mode `Url` a l'enum et ecrire une methode qui simule le chargement d'une description depuis une URL (affichez simplement l'URL que vous utiliseriez).

**Indices** :
- # Etape 1 : Ajoutez la valeur `Url` dans l'enum `InformationMode`
- # Etape 2 : Completez la methode `FetchDescriptionFromUrl` qui prend une URL et retourne un message indiquant l'URL qui serait chargee
- # Indice : En production, on utiliserait `HttpClient.GetStringAsync(url)`, ici contentez-vous de simuler le comportement

In [1]:
// Exercice : Ajouter un mode de fourniture URL
// 1. Ajoutez la valeur Url a l'enum InformationMode
// 2. Completez la methode de simulation

// TODO etudiant : ajoutez Url a l'enum ci-dessus (dans la cellule precedente)
// public enum InformationMode { Variable, Prompt, File, Url }

string FetchDescriptionFromUrl(string url)
{
    // TODO etudiant : simulez le chargement depuis l'URL
    // En production : var content = await new HttpClient().GetStringAsync(url);
    return null;  // TODO etudiant : retournez un message simulant le chargement
}

// Test
var simulated = FetchDescriptionFromUrl("https://example.com/task-description.txt");
Console.WriteLine(simulated ?? "Exercice a completer");

None

### 2. Recueil des informations

Selon le mode de fourniture des informations choisi, on récupère la tâche à accomplir dans le notebook de travail.

In [5]:
var infoCollectionDisplay = display("Collecte d'informations en cours...");

var mode = InformationMode.Variable;

string taskDescription = "Créer un notebook .Net interactive permettant de requêter DBPedia, utilisant les package Nuget dotNetRDF et XPlot.Plotly.Interactive. Commencer par éditer les cellules de Markdown pour définir et affiner l'exemple de requête dont le graphique final sera le plus pertinent. Choisir un exemple de requête complexe, manipulant des aggrégats, qui pourront être synthétisés dans un graphique. Une fois les cellules de code alimentées et les bugs corrigés, s'assurer que la sortie de la cellule générant le graphique est conforme et pertinente.";

if (mode == InformationMode.Variable)
{
    display("Utilisation de la variable pour la description de la tâche.");
}
else if (mode == InformationMode.Prompt)
{
    var questions = new[]
    {
        "Bonjour! Veuillez fournir une brève description de la tâche à accomplir.",
        "Quels sont les principaux objectifs de cette tâche?",
        "Y a-t-il des contraintes ou des conditions spécifiques à prendre en compte?",
        "Des informations supplémentaires que vous souhaitez ajouter?"
    };

    taskDescription = string.Empty;
    foreach (var question in questions)
    {
        var response = await IKernel.GetInputAsync(question);
        taskDescription += $"{question}\\n{response}\\n\\n";
    }
}


display("Informations recueillies :\\n" + taskDescription);

### Exercice : Valider la description de la tache

Avant de lancer la generation du notebook, il est utile de verifier que la description fournie est suffisamment precise pour guider le modele.

**Objectif** : Completez la methode `ValidateTaskDescription` qui verifie que la description contient au moins 10 mots, mentionne au moins un package NuGet ou une technologie, et retourne un message de validation ou d'erreur.

**Indices** :
- # Etape 1 : Comptez le nombre de mots avec `description.Split(' ', StringSplitOptions.RemoveEmptyEntries).Length`
- # Etape 2 : Cherchez des mots cles techniques (package, NuGet, using, requete, graphique, etc.) avec `description.Contains()`
- # Etape 3 : Retournez un tuple `(bool isValid, string message)` indiquant le resultat
- # Indice : Utilisez une liste de mots cles et LINQ `.Any(k => description.Contains(k))` pour la detection

In [1]:
// Exercice : Valider la description de la tache
// Completez la methode pour verifier la qualite de la description

(bool IsValid, string Message) ValidateTaskDescription(string description)
{
    // TODO etudiant : verifiez que la description contient au moins 10 mots
    // TODO etudiant : cherchez des mots cles techniques
    // TODO etudiant : retournez un tuple (isValid, message)
    return (false, "Exercice a completer");  // TODO etudiant : remplacez par l'implementation
}

// Test avec la description actuelle
var validation = ValidateTaskDescription(taskDescription);
Console.WriteLine($"Validation : {validation.IsValid} - {validation.Message}");

None

### 3. Personnalisation du Notebook de Travail

On charge un notebook template contenant des parties de Markdown et de code à compléter, et on injecte la tâche dans la partie descriptive en entête du notebook .Net interactive.

In [6]:

var notebookPath = @$"./Workbooks/Workbook-{DateTime.Now.ToFileTime()}.ipynb";


### Exercice : Generer un nom de notebook personnalise

Le chemin du notebook est actuellement genere avec un timestamp. Vous pouvez enrichir cette logique pour produire un nom plus descriptif.

**Objectif** : Completez la methode `GenerateNotebookName` qui construit un nom de fichier incluant un prefix fourni, la date du jour et un identifiant court derive de la description de la tache.

**Indices** :
- # Etape 1 : Extrayez les 3 premiers mots significatifs de `taskDescription` (supprimez les mots vides comme "un", "de", "le", "la", "des")
- # Etape 2 : Formatez la date avec `DateTime.Now.ToString("yyyy-MM-dd")`
- # Etape 3 : Concatenez prefix + date + mots extraits avec des underscores et ajoutez l'extension `.ipynb`
- # Indice : Utilisez `string.Join("_", mots.Take(3))` pour assembler les mots cles

In [1]:
// Exercice : Generer un nom de notebook personnalise
// Completez la methode pour construire un nom de fichier descriptif

string GenerateNotebookName(string prefix, string description)
{
    // TODO etudiant : extrayez les mots significatifs de la description
    // TODO etudiant : filtrez les mots vides (un, de, le, la, des, du, etc.)
    // TODO etudiant : formatez la date et assemblez le nom
    return null;  // TODO etudiant : remplacez par l'implementation
}

// Test (decommentez apres implementation)
// var customName = GenerateNotebookName("SK", taskDescription);
// Console.WriteLine($"Nom genere : {customName}");
Console.WriteLine("Exercice a completer");

None

### Exécution et Mise à Jour Itérative avec AutoInvokeSKAgentsNotebookUpdater


In [7]:
var logger = new DisplayLogger("NotebookUpdater", LogLevel.Trace);
var updater = new AutoInvokeSKAgentsNotebookUpdater(notebookPath, logger);
updater.SetStartingNotebookFromTemplate(taskDescription);
display("Appel à UpdateNotebookWithAutoInvokeSKAgents...");
await updater.UpdateNotebookWithAutoInvokeSKAgents();
display("Mise à jour du notebook terminée.");


## Conclusion

Ce notebook illustre un cas d'usage **méta** de Semantic-Kernel : non pas interroger un LLM pour générer du texte, mais l'orchestrer pour **concevoir et mettre à jour d'autres notebooks .NET Interactive** de bout en bout.

**Ce que démontre l'arc** :

- **Initialisation** : combinaison des packages de manipulation de notebooks et du kernel Semantic-Kernel, en distinguant soigneusement le kernel d'exécution .NET Interactive du kernel SK.
- **Fourniture d'informations** : un `enum` `InformationMode` (Variable, Prompt, File) permet de décrire la tâche du notebook cible de plusieurs façons — exercice d'extension vers un mode `Url`.
- **Personnalisation** : chargement d'un notebook template (Markdown + code à compléter), injection de la description dans l'en-tête, génération d'un nom de fichier — exercice d'enrichissement du nom.
- **Exécution itérative** : `AutoInvokeSKAgentsNotebookUpdater` orchestre le cycle complet (template → description → mise à jour → exécution auto-invoquée) en s'appuyant sur le **function calling** OpenAI pour appliquer les modifications proposées par le LLM.

**Points clés** :

- Le notebook est **auto-modifiant** : il utilise l'API .NET Interactive pour réécrire et ré-exécuter un autre notebook — un pattern d'**agent génératif** où le LLM agit sur un artefact de code, pas seulement sur du texte.
- La séparation **kernel d'exécution** vs **kernel SK** est essentielle : elle isole l'orchestration du LLM de l'exécution du code généré.
- Les exercices (`InformationMode`, `GenerateNotebookName`) ancrent la pratique sur les deux briques d'extension les plus naturelles : élargir les sources d'entrée et enrichir la personnalisation de sortie.

**Référence** : Microsoft Semantic-Kernel — [Auto Function Calling](https://learn.microsoft.com/en-us/semantic-kernel/concepts/ai-services/chat-completion/function-calling). Le notebook pousse le function calling jusqu'à l'auto-édition de notebooks, anticipant les patterns d'agents logiciels qui modifient leur propre runtime.